# Sleep Disorder Risk Predictor

**Goal:** Build a small machine learning classifier that predicts a person's sleep disorder risk category from lifestyle, health, and sleep-related features.

This notebook includes:

1. Data loading and basic inspection  
2. Exploratory data analysis  
3. Data preprocessing  
4. Model training and comparison  
5. Hyperparameter tuning  
6. Final model evaluation  
7. Feature importance  
8. Example prediction for a new person  
9. Save the finalized model

**Target variable:** `sleep_disorder_risk`


In [ ]:
import sys
!{sys.executable} -m pip install numpy pandas scikit-learn matplotlib seaborn


In [ ]:
import numpy as np
print(np.__version__)


## 1. Import Libraries


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)


## 2. Load the Dataset

The CSV file should be in the same folder as this notebook. If needed, change the file path below.


In [ ]:
DATA_PATH = "sleep_health_dataset.csv"
GITHUB_DATA_URL = "https://raw.githubusercontent.com/martini-i/sleep-risk-predictor/main/sleep_health_dataset.csv"

try:
    df = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    # Colab/GitHub fallback for reproducible runs.
    df = pd.read_csv(GITHUB_DATA_URL)

print("Dataset shape:", df.shape)
df.head()


## 3. Basic Data Inspection


In [ ]:
df.info()


In [ ]:
df.describe(include="all").T


In [ ]:
# Check missing values
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]


## 4. Target Variable Distribution

This shows whether the classes are balanced or imbalanced.


In [ ]:
target_col = "sleep_disorder_risk"

class_counts = df[target_col].value_counts()
class_percentages = df[target_col].value_counts(normalize=True).mul(100).round(2)

target_summary = pd.DataFrame({
    "count": class_counts,
    "percentage": class_percentages
})

target_summary


In [ ]:
class_counts.plot(kind="bar")
plt.title("Sleep Disorder Risk Class Distribution")
plt.xlabel("Risk Category")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()


## 5. Exploratory Data Analysis

These quick charts help show how key features relate to sleep disorder risk.


In [ ]:
# Numeric feature distributions by target category
important_numeric_features = [
    "sleep_duration_hrs",
    "sleep_quality_score",
    "stress_score",
    "screen_time_before_bed_mins",
    "caffeine_mg_before_bed",
    "heart_rate_resting_bpm",
    "cognitive_performance_score"
]

for col in important_numeric_features:
    if col in df.columns:
        df.boxplot(column=col, by=target_col, grid=False)
        plt.title(f"{col} by Sleep Disorder Risk")
        plt.suptitle("")
        plt.xlabel("Risk Category")
        plt.ylabel(col)
        plt.show()


In [ ]:
# Correlation heatmap for numeric columns
numeric_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(12, 8))
plt.imshow(numeric_df.corr(), aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(numeric_df.columns)), numeric_df.columns, rotation=90)
plt.yticks(range(len(numeric_df.columns)), numeric_df.columns)
plt.title("Correlation Heatmap for Numeric Features")
plt.tight_layout()
plt.show()


## 6. Choose Features and Target

`person_id` is removed because it is just an identifier.

There are two possible project versions:

- **Full-feature version:** uses all non-ID features. This may produce high accuracy because some columns, such as `sleep_quality_score`, `cognitive_performance_score`, and `felt_rested`, may be closely related to the target.
- **No-leakage version:** removes features that may be too directly connected to the target. This is often better for a realistic prediction project.

This notebook uses the **no-leakage version**.


In [ ]:
# Columns that should not be model inputs
id_columns = ["person_id"]

# These are potentially too directly related to sleep disorder risk.
potential_leakage_columns = [
    "sleep_quality_score",
    "cognitive_performance_score",
    "felt_rested"
]

# Use no-leakage features by default
columns_to_drop = id_columns + potential_leakage_columns + [target_col]
columns_to_drop = [col for col in columns_to_drop if col in df.columns]

X = df.drop(columns=columns_to_drop)
y = df[target_col]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Features used:")
print(list(X.columns))


## 7. Train/Test Split

`stratify=y` keeps the class proportions similar in the training and test sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)


## 8. Preprocessing Pipeline

The pipeline:

- Fills missing numeric values with the median
- Scales numeric features
- Fills missing categorical values with the most common value
- One-hot encodes categorical features


In [ ]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## 9. Train Baseline Models

We compare two models:

1. **Logistic Regression:** simple and explainable baseline  
2. **Random Forest:** stronger nonlinear model


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        max_depth=None,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

results = []
fitted_pipelines = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    results.append({"model": name, "accuracy": acc})
    fitted_pipelines[name] = pipeline

results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
results_df


## 10. Fine Tune the Model

This section tunes a Random Forest with GridSearchCV and compares the tuned result against the best baseline model.


In [ ]:
baseline_best_model_name = results_df.iloc[0]["model"]
baseline_best_pipeline = fitted_pipelines[baseline_best_model_name]
baseline_best_accuracy = results_df.iloc[0]["accuracy"]

tuning_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

grid_search = GridSearchCV(
    estimator=tuning_pipeline,
    param_grid=param_grid,
    scoring="f1_weighted",
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)
tuned_pipeline = grid_search.best_estimator_

tuned_pred = tuned_pipeline.predict(X_test)
tuned_accuracy = accuracy_score(y_test, tuned_pred)

print("Best baseline model:", baseline_best_model_name)
print("Best baseline accuracy:", round(baseline_best_accuracy, 4))
print("Best tuned params:", grid_search.best_params_)
print("Best CV f1_weighted:", round(grid_search.best_score_, 4))
print("Tuned test accuracy:", round(tuned_accuracy, 4))

if tuned_accuracy >= baseline_best_accuracy:
    best_model_name = "Random Forest (Tuned)"
    best_pipeline = tuned_pipeline
else:
    best_model_name = baseline_best_model_name
    best_pipeline = baseline_best_pipeline

print("Selected final model:", best_model_name)


## 11. Evaluate the Final Model


In [ ]:
if "best_pipeline" not in globals():
    best_model_name = results_df.iloc[0]["model"]
    best_pipeline = fitted_pipelines[best_model_name]

print("Best model:", best_model_name)

y_pred = best_pipeline.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Classification Report:")
print(classification_report(y_test, y_pred))


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, xticks_rotation=45)
plt.title(f"Confusion Matrix: {best_model_name}")
plt.tight_layout()
plt.show()


## 12. Feature Importance

Permutation importance estimates how much model performance drops when each feature is randomly shuffled.

To keep the notebook fast, this section uses a sample of the test set.


In [ ]:
# Use a smaller sample for speed
sample_size = min(5000, len(X_test))
X_test_sample = X_test.sample(sample_size, random_state=RANDOM_STATE)
y_test_sample = y_test.loc[X_test_sample.index]

perm = permutation_importance(
    best_pipeline,
    X_test_sample,
    y_test_sample,
    n_repeats=5,
    random_state=RANDOM_STATE,
    scoring="accuracy"
)

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

importance_df.head(15)


In [ ]:
top_n = 15
plot_df = importance_df.head(top_n).sort_values("importance_mean")

plt.figure(figsize=(8, 6))
plt.barh(plot_df["feature"], plot_df["importance_mean"])
plt.xlabel("Mean Decrease in Accuracy")
plt.title(f"Top {top_n} Most Important Features")
plt.tight_layout()
plt.show()


## 13. Predict Risk for a New Person

This creates one example row using median/mode values from the dataset, then modifies a few inputs.


In [ ]:
# Start with a typical person based on the training data
example = X_train.iloc[[0]].copy()

# Modify a few values to create a custom example
updates = {
    "age": 35,
    "bmi": 27.5,
    "sleep_duration_hrs": 5.5,
    "sleep_latency_mins": 30,
    "wake_episodes_per_night": 4,
    "caffeine_mg_before_bed": 120,
    "screen_time_before_bed_mins": 90,
    "stress_score": 7.5,
    "work_hours_that_day": 9.0,
    "shift_work": 0,
    "exercise_day": 0
}

for col, value in updates.items():
    if col in example.columns:
        example.loc[example.index[0], col] = value

prediction = best_pipeline.predict(example)[0]
probabilities = best_pipeline.predict_proba(example)[0]
classes = best_pipeline.classes_

print("Predicted sleep disorder risk:", prediction)

pd.DataFrame({
    "risk_category": classes,
    "predicted_probability": probabilities
}).sort_values("predicted_probability", ascending=False)


## 14. Save the Final Model

This saves the trained pipeline so it can be reused later for inference or deployment.


In [ ]:
MODEL_PATH = "sleep_risk_model.joblib"
joblib.dump(best_pipeline, MODEL_PATH)

print("Saved model to:", os.path.abspath(MODEL_PATH))


## 15. Conclusion

> This project built a machine learning classifier to predict sleep disorder risk using lifestyle, health, and sleep behavior data. After preprocessing numeric and categorical features, two models were compared: Logistic Regression and Random Forest. The best-performing model was further tuned with GridSearchCV, evaluated using accuracy, precision, recall, F1-score, and a confusion matrix, and then saved for reuse. Feature importance analysis suggested which factors had the strongest effect on predicted sleep risk. To avoid overly optimistic performance, potentially leaky features such as sleep quality score, cognitive performance score, and felt rested were removed from the main model.
